<img src="./images/hsph.png" alt="HSPH Logo" width="400"><br>

# Lab 2.3 — RAG with ChromaDB (Persistent Vector Store)

In this notebook, you will build a simple Retrieval-Augmented Generation (RAG) workflow using:

- **MiniLM embeddings** (local)
- **ChromaDB** (persistent vector store)
- **Ollama** (local LLM)

### Learning goals
1) Store note embeddings persistently (so you don’t rebuild every time)
2) Retrieve notes relevant to a dementia-related query
3) Use retrieved notes to extract a **dementia phenotype (Yes/No)** with evidence

> Important: We do **not** pass gold labels into retrieval or prompting. Gold labels are used only later for evaluation.

## 1. Prepare Notes Data for Embedding

We will load the cleaned notes dataset produced in Lab 6:

- `lab1_clean_notes_baseline.parquet`

This dataset is already:
- restricted to the chosen 2-year window
- deduplicated across note versions
- reduced to the most recent note per (patient, visit, note type)

In [1]:
# -----------------------------------------------------------
# 1.1. Load Cleaned Notes from Lab 6 (Parquet)
# -----------------------------------------------------------

from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

filepath = Path("data_files")

notes_df = pd.read_parquet(filepath / "lab1_clean_notes_baseline.parquet")

print("Notes loaded:", notes_df.shape)
display(notes_df.head(10))

Notes loaded: (2507, 7)


,patient_num,visit_id,note_id,note_csn_id,inpatient_note_type_dsc,create_dts_shifted,note_txt_deid
0,H120513333,6297755287,2419560135,2305518755,Progress Notes,2017-01-01 16:08:00,Physical therapy progress note for visit #2. T...
1,H111336931,6332144391,2829946925,2712892487,Progress Notes,2017-01-01 17:11:00,The patient was seen for follow-up regarding s...
2,H120897999,6358002173,3208794021,3103021255,Progress Notes,2017-01-01 18:30:00,This is a progress note for a 69-year-old woma...
3,H120897999,6358002173,3210662987,3104938425,Assessment & Plan Note,2017-01-02 17:58:00,Assessment & Plan:\n\nThe patient has a histor...
4,H122074591,6270644869,2578637669,2455733125,Progress Notes,2017-01-03 13:56:00,This is an update on an 85-year-old man with a...
5,H122355386,6283144543,2149720299,2050365293,Progress Notes,2017-01-03 17:32:00,A message was left for the patient requesting ...
6,H120189926,6346141193,2984884099,2872365651,Assessment & Plan Note,2017-01-03 18:39:00,The patient was evaluated for blood pressure m...
7,H120189926,6346141193,2984887107,2872410355,Progress Notes,2017-01-03 18:40:00,The patient was seen for evaluation and manage...
8,H113566534,6313898129,2700402247,2579340567,Progress Notes,2017-01-04 14:40:00,Interval History:\nThe patient returned for fo...
9,H113383484,6341517751,3104765319,2995799951,Progress Notes,2017-01-04 17:31:00,The patient presented for a follow-up evaluati...


## 2. Embed and Store Entire Clinical Notes in ChromaDB

In this step, we process and store each clinical note as a **complete document** in a ChromaDB vector store. This approach preserves full patient context, making it ideal for semantic search and downstream clinical reasoning.

### Step 2.1: Embed and Store Notes
1. **Embed Full Notes**
   Each note is converted into a semantic vector using a lightweight transformer model (`MiniLM`).

2. **Store in ChromaDB**
   The embedded vector and its associated metadata (e.g., patient ID, encounter number, visit date) are stored together in a persistent ChromaDB directory.

### Why This Approach?

Storing full notes is especially useful when:
- You want to retrieve the complete narrative for clinical context
- Your downstream task (e.g., summarization or decision support) depends on comprehensive input
- Each note fits within the input limit of a single LLM call

This setup supports more faithful summarization and reasoning than chunk-based approaches when notes are relatively short and self-contained.

<img src="./images/rag_full.png" alt="RAG Full" width="900">


In [2]:
# -----------------------------------------------------------
# 2.1. Embed Notes Using MiniLM and Store in Persistent ChromaDB
# -----------------------------------------------------------

# Note for students:
# This notebook uses a public Hugging Face model.
# You do NOT need a Hugging Face account or token for this exercise.
# If you see a warning about HF_TOKEN, it can be ignored.

import os, logging
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


# PERSIST_DIR = "./data_files/chroma_db_notes_epi264"
PERSIST_DIR = (filepath / "chroma_db_notes_epi264")


# Silence Hugging Face Hub warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

# Initialize embedding model (local)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create/load Chroma store
vectorstore = Chroma(
    persist_directory=PERSIST_DIR,
    embedding_function=embedding_model
)

# NOTE: If you re-run this cell, you may add duplicates.
# For the workshop: we will only add texts if the store is empty.

existing_count = vectorstore._collection.count()
print("Existing vectors in Chroma:", existing_count)

if existing_count == 0:
    documents = notes_df["note_txt_deid"].fillna("").tolist()
    metadata = notes_df[[
        "patient_num",
        "visit_id",
        "note_csn_id",
        "inpatient_note_type_dsc"
    ]].to_dict(orient="records")

    vectorstore.add_texts(texts=documents, metadatas=metadata)
    print(f"✅ Embedded and stored {len(documents)} notes in ChromaDB.")
else:
    print("✅ ChromaDB already populated. Skipping add_texts to avoid duplicates.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Existing vectors in Chroma: 0
✅ Embedded and stored 2507 notes in ChromaDB.


## 3. Retrieving Clinical Notes with Similarity Score (RAG Retrieval with ChromaDB)

In this section, we retrieve relevant clinical notes from a ChromaDB vector store using vector similarity techniques. We explore both standard and advanced retrieval strategies to improve the relevance and diversity of retrieved context for downstream LLM generation.

### Key Retrieval Strategies

1. **Define a Clinical Query (Step 3.1)**
   - The user provides a natural language question (e.g., "Who has asthma and is taking Fluticasone and Albuterol?").

2. **Similarity Search with Scores (Step 3.2)**
   - Retrieves the top-k notes ranked by semantic similarity to the query.
   - Returns cosine-based relevance scores for each match.

3. **Score Threshold Filtering (Step 3.3)**
   - Filters out low-confidence matches using a minimum similarity score.
   - Retains only notes that meet a defined relevance threshold (e.g., ≥ 0.6).

4. **Maximal Marginal Relevance (MMR) Search (Step 3.4)**
   - Balances similarity and diversity.
   - Reduces redundancy by selecting a diverse subset of highly relevant notes.

### Why Use These Strategies?

High-quality retrieval is critical to the success of RAG workflows. These techniques help:
- Improve the contextual relevance of inputs to the LLM
- Filter out irrelevant or low-confidence documents
- Encourage diverse results to reduce bias and improve robustness

<img src="./images/rag_retrieval.png" alt="RAG Retrieval" width="900">


In [3]:
# -----------------------------------------------------------
# 3.1. Define the Query for Clinical Note Retrieval
# -----------------------------------------------------------
# This cell defines a natural language query to search the embedded clinical notes
# stored in ChromaDB using semantic similarity.

# Concept:
# The query will be *** automatically embedded *** by the vectorstore before performing
# a semantic comparison against stored note vectors.
# -----------------------------------------------------------

# Example Clinical Query:
query = "Does this patient have a documented diagnosis of dementia or Alzheimer disease?"

# Display the query for reference
display(Markdown(f"### 🔍 Query: `{query}`"))


### 🔍 Query: `Does this patient have a documented diagnosis of dementia or Alzheimer disease?`

In [4]:
# -----------------------------------------------------------
# 3.2. Perform Similarity Search with Relevance Scores
# -----------------------------------------------------------
# This cell retrieves the top-k clinical notes most semantically similar to the input query.
# Each result includes a cosine similarity score returned by LangChain's Chroma vectorstore wrapper.

# Function used:
# - vectorstore.similarity_search_with_relevance_scores(query, k=10)
#   Returns a list of (Document, score) tuples.

# Relevance Score Interpretation (heuristic only):
# - 0.90 – 1.00 → Highly relevant
# - 0.70 – 0.90 → Strong relevance
# - 0.50 – 0.70 → Moderate relevance
# - 0.30 – 0.50 → Low relevance
# - 0.00 – 0.30 → Minimal or no relevance
# -----------------------------------------------------------

# Perform the similarity search
results = vectorstore.similarity_search_with_relevance_scores(query, k=10)

# Display the results
display(Markdown("### 🔍 Retrieved Notes with Relevance Scores"))

for idx, (doc, score) in enumerate(results, 1):
    meta = doc.metadata or {}
    excerpt = (doc.page_content or "")[:900].replace("\n", " ")

    display(Markdown(
        f"---\n**Result {idx}**  \n"
        f"- **Relevance Score:** `{score:.4f}`  \n"
        f"- **patient_num:** `{meta.get('patient_num','N/A')}`  \n"
        f"- **visit_id:** `{meta.get('visit_id','N/A')}`  \n"
        f"- **note_csn_id:** `{meta.get('note_csn_id','N/A')}`  \n"
        f"- **note_type:** `{meta.get('inpatient_note_type_dsc','N/A')}`  \n"
        f"**Excerpt:**\n```text\n{excerpt}...\n```"
    ))


### 🔍 Retrieved Notes with Relevance Scores

---
**Result 1**  
- **Relevance Score:** `0.4943`  
- **patient_num:** `H112686715`  
- **visit_id:** `6447865553`  
- **note_csn_id:** `3925256583`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
A physician summary form including the medication list and diagnosis was sent on 8/24/2017. The documentation provides confirmation of a prior diagnosis, which is consistent with a working diagnosis of dementia....
```

---
**Result 2**  
- **Relevance Score:** `0.4901`  
- **patient_num:** `H115124574`  
- **visit_id:** `6382479701`  
- **note_csn_id:** `3225668549`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
On 10/19/2017, the patient was admitted as an inpatient. Contact was successfully made with the patient and an email was sent to the inpatient case manager regarding the admission. The primary medical concern noted at the time of communication was congestive heart failure. Of note, there is a prior working diagnosis of dementia that may inform ongoing management considerations....
```

---
**Result 3**  
- **Relevance Score:** `0.4832`  
- **patient_num:** `H115124574`  
- **visit_id:** `6368409967`  
- **note_csn_id:** `3075125799`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
The patient was admitted for observation with the initial assessment taking place on 8/18/2017. There was successful communication with the inpatient care manager regarding the admission, and congestive heart failure was documented as a relevant concern. Notably, there is a history suggesting possible cognitive impairment consistent with a prior diagnosis of dementia. No additional interval events or updates were provided....
```

---
**Result 4**  
- **Relevance Score:** `0.4824`  
- **patient_num:** `H113694308`  
- **visit_id:** `6412697495`  
- **note_csn_id:** `3578176631`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
The patient was observed sitting upright in bed and appeared comfortable. Vital signs were stable while breathing room air. No complaints of pain were reported at this time, but a cough was noted. A chest x-ray was performed at the bedside. The patient was provided with an update about the plan of care. Additional information can be found in the chart and recent orders.  A prior diagnosis of dementia continues to be considered in the ongoing assessment....
```

...

---
**Result 6**  
- **Relevance Score:** `0.4772`  
- **patient_num:** `H117604331`  
- **visit_id:** `6403897423`  
- **note_csn_id:** `3470839149`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
Today, the patient was seen in person for a follow-up visit and to receive a flu vaccination. Reports from her daughter indicate that there have been no new concerns and her condition remains stable. The patient continues to receive assistance at home with her activities of daily living. The care team will remain available for ongoing support and plans to check in by phone in approximately 3 months. There is a history of cognitive impairment consistent with a highly confident diagnosis of dementia....
```

---
**Result 7**  
- **Relevance Score:** `0.4640`  
- **patient_num:** `H116088887`  
- **visit_id:** `6387926043`  
- **note_csn_id:** `3286513763`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
A recent outreach call was made to the patient to check in, and a voice mail message was left. Coordination was discussed with the social worker. Continued monitoring is recommended, given the established diagnosis of dementia....
```

---
**Result 8**  
- **Relevance Score:** `0.4639`  
- **patient_num:** `H116882795`  
- **visit_id:** `6353841785`  
- **note_csn_id:** `2927596265`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
Patient was evaluated during this admission. Plans previously outlined were reviewed and found appropriate. Examination was performed, and the care provided remains in alignment with prior assessments. The patient's history is notable for a prior diagnosis of dementia....
```

---
**Result 9**  
- **Relevance Score:** `0.4547`  
- **patient_num:** `H114980041`  
- **visit_id:** `6341343803`  
- **note_csn_id:** `2916158553`  
- **note_type:** `Assessment & Plan Note`  
**Excerpt:**
```text
There have been no recent diagnostic tests. The plan is to obtain Humphrey Visual Field testing and optical coherence tomography of the optic nerves to further evaluate the patient's condition, which includes a prior diagnosis of dementia....
```

---
**Result 10**  
- **Relevance Score:** `0.4498`  
- **patient_num:** `H115124574`  
- **visit_id:** `6404371571`  
- **note_csn_id:** `3475777617`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
On 01/11/2018, the patient was admitted through the emergency department. Shortness of breath was noted at the time of admission. Communication with the inpatient care management team was successfully established. While reviewing the chart, a prior diagnosis of dementia was considered to be part of the patient's history....
```

In [5]:
# -----------------------------------------------------------
# 3.3. Use a Retriever with a Similarity Score Threshold
# -----------------------------------------------------------
# This cell configures a retriever that only returns documents whose
# cosine similarity scores exceed a predefined threshold.

# Parameters:
# - search_type="similarity_score_threshold": Enables threshold-based filtering.
# - search_kwargs={"k": 10, "score_threshold": 0.6}
#     - k: Maximum number of documents to *evaluate* (not necessarily return)
#     - score_threshold: Minimum similarity score (0–1) required for inclusion

# Purpose:
# Improves retrieval precision by excluding weak semantic matches—
# especially important for clinical reasoning and safety-critical applications.
# -----------------------------------------------------------

# Define threshold and top-k limit
# -----------------------------------------------------------
# 3.3. Retriever with Similarity Score Threshold
# -----------------------------------------------------------

score_threshold = 0.47
top_k = 10

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": top_k, "score_threshold": score_threshold}
)

threshold_docs = retriever.invoke(query)

display(Markdown(f"### 🔎 Retrieved Notes (Score ≥ {score_threshold})"))

for idx, doc in enumerate(threshold_docs, 1):
    meta = doc.metadata or {}
    excerpt = (doc.page_content or "")[:900].replace("\n", " ")

    display(Markdown(
        f"---\n**Doc {idx}**  \n"
        f"- **patient_num:** `{meta.get('patient_num','N/A')}`  \n"
        f"- **visit_id:** `{meta.get('visit_id','N/A')}`  \n"
        f"- **note_csn_id:** `{meta.get('note_csn_id','N/A')}`  \n"
        f"- **note_type:** `{meta.get('inpatient_note_type_dsc','N/A')}`  \n"
        f"**Excerpt:**\n```text\n{excerpt}...\n```"
    ))

display(Markdown(f"**✅ Total returned:** `{len(threshold_docs)}`"))


### 🔎 Retrieved Notes (Score ≥ 0.47)

---
**Doc 1**  
- **patient_num:** `H112686715`  
- **visit_id:** `6447865553`  
- **note_csn_id:** `3925256583`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
A physician summary form including the medication list and diagnosis was sent on 8/24/2017. The documentation provides confirmation of a prior diagnosis, which is consistent with a working diagnosis of dementia....
```

---
**Doc 2**  
- **patient_num:** `H115124574`  
- **visit_id:** `6382479701`  
- **note_csn_id:** `3225668549`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
On 10/19/2017, the patient was admitted as an inpatient. Contact was successfully made with the patient and an email was sent to the inpatient case manager regarding the admission. The primary medical concern noted at the time of communication was congestive heart failure. Of note, there is a prior working diagnosis of dementia that may inform ongoing management considerations....
```

---
**Doc 3**  
- **patient_num:** `H115124574`  
- **visit_id:** `6368409967`  
- **note_csn_id:** `3075125799`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
The patient was admitted for observation with the initial assessment taking place on 8/18/2017. There was successful communication with the inpatient care manager regarding the admission, and congestive heart failure was documented as a relevant concern. Notably, there is a history suggesting possible cognitive impairment consistent with a prior diagnosis of dementia. No additional interval events or updates were provided....
```

---
**Doc 4**  
- **patient_num:** `H113694308`  
- **visit_id:** `6412697495`  
- **note_csn_id:** `3578176631`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
The patient was observed sitting upright in bed and appeared comfortable. Vital signs were stable while breathing room air. No complaints of pain were reported at this time, but a cough was noted. A chest x-ray was performed at the bedside. The patient was provided with an update about the plan of care. Additional information can be found in the chart and recent orders.  A prior diagnosis of dementia continues to be considered in the ongoing assessment....
```

---
**Doc 5**  
- **patient_num:** `H117604331`  
- **visit_id:** `6383967105`  
- **note_csn_id:** `3233558851`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
A telephone call was made to the patient's spouse, who reported that since her most recent visit with her physician, there have been no new concerns. No recent falls were noted. The patient continues to receive daily assistance from private home health aides with her activities of daily living, consistent with care needs in the setting of a prior diagnosis of dementia. The patient has a follow-up appointment scheduled in December, with plans for additional care management contact at that time. The family has been encouraged to reach out if any issues arise before then....
```

---
**Doc 6**  
- **patient_num:** `H117604331`  
- **visit_id:** `6403897423`  
- **note_csn_id:** `3470839149`  
- **note_type:** `Progress Notes`  
**Excerpt:**
```text
Today, the patient was seen in person for a follow-up visit and to receive a flu vaccination. Reports from her daughter indicate that there have been no new concerns and her condition remains stable. The patient continues to receive assistance at home with her activities of daily living. The care team will remain available for ongoing support and plans to check in by phone in approximately 3 months. There is a history of cognitive impairment consistent with a highly confident diagnosis of dementia....
```

**✅ Total returned:** `6`

In [6]:
# -----------------------------------------------------------
# 3.4. Perform Maximal Marginal Relevance (MMR) Search
# -----------------------------------------------------------
# This cell retrieves clinical notes using Maximal Marginal Relevance (MMR),
# an approach that balances query relevance with result diversity.

# Parameters:
# - fetch_k = 500: Number of top-ranked candidates to evaluate before reranking
# - k = 5: Final number of diverse, relevant documents to return
# - lambda_mult:
#     - 0.0 → prioritize diversity (minimal redundancy)
#     - 1.0 → prioritize similarity to the query
#     - 0.5 → balanced trade-off between diversity and relevance

# Purpose:
# MMR reduces redundancy by penalizing near-duplicate documents, while still
# prioritizing clinical notes that are highly relevant to the query.
# This is especially useful in settings where multiple perspectives or
# treatment variations are valuable (e.g., different asthma management plans).
# -----------------------------------------------------------

# Perform MMR search
results = vectorstore.max_marginal_relevance_search(
    query=query,
    k=5,
    fetch_k=500,
    lambda_mult=0.8
)

# Display results
display(Markdown("### 📘 Retrieved Clinical Notes Using Maximal Marginal Relevance (MMR)"))

for idx, doc in enumerate(results, 1):
    patient = doc.metadata.get("patient_num", "N/A")
    date = doc.metadata.get("start_date", "N/A")
    doc_id = getattr(doc, "id", "N/A")
    excerpt = doc.page_content[:1000].replace("\n", " ")

    display(Markdown(
        f"**Document {idx}**  \n"
        f"- **Patient Num:** `{patient}`  \n"
        f"- **Document ID:** `{doc_id}`  \n\n"
        f"**Excerpt:**\n```text\n{excerpt}...\n```"
    ))

display(Markdown(f"**✅ Total MMR Results Returned:** `{len(results)}`"))


### 📘 Retrieved Clinical Notes Using Maximal Marginal Relevance (MMR)

**Document 1**  
- **Patient Num:** `H112686715`  
- **Document ID:** `bd5563fa-44ee-438a-95d3-14aeb17700a1`  

**Excerpt:**
```text
A physician summary form including the medication list and diagnosis was sent on 8/24/2017. The documentation provides confirmation of a prior diagnosis, which is consistent with a working diagnosis of dementia....
```

**Document 2**  
- **Patient Num:** `H115124574`  
- **Document ID:** `5781faf3-e486-4748-b9c2-d92c8113767d`  

**Excerpt:**
```text
The patient was admitted for observation with the initial assessment taking place on 8/18/2017. There was successful communication with the inpatient care manager regarding the admission, and congestive heart failure was documented as a relevant concern. Notably, there is a history suggesting possible cognitive impairment consistent with a prior diagnosis of dementia. No additional interval events or updates were provided....
```

**Document 3**  
- **Patient Num:** `H113694308`  
- **Document ID:** `3b2e3a32-9f26-4a04-8b19-5b6947998e39`  

**Excerpt:**
```text
The patient was observed sitting upright in bed and appeared comfortable. Vital signs were stable while breathing room air. No complaints of pain were reported at this time, but a cough was noted. A chest x-ray was performed at the bedside. The patient was provided with an update about the plan of care. Additional information can be found in the chart and recent orders.  A prior diagnosis of dementia continues to be considered in the ongoing assessment....
```

**Document 4**  
- **Patient Num:** `H117604331`  
- **Document ID:** `5270a643-84dc-4100-a2aa-38af76fe60bb`  

**Excerpt:**
```text
Today, the patient was seen in person for a follow-up visit and to receive a flu vaccination. Reports from her daughter indicate that there have been no new concerns and her condition remains stable. The patient continues to receive assistance at home with her activities of daily living. The care team will remain available for ongoing support and plans to check in by phone in approximately 3 months. There is a history of cognitive impairment consistent with a highly confident diagnosis of dementia....
```

**Document 5**  
- **Patient Num:** `H114980041`  
- **Document ID:** `907db9ba-ad03-4c32-9c5f-1a2d5fd093ca`  

**Excerpt:**
```text
There have been no recent diagnostic tests. The plan is to obtain Humphrey Visual Field testing and optical coherence tomography of the optic nerves to further evaluate the patient's condition, which includes a prior diagnosis of dementia....
```

**✅ Total MMR Results Returned:** `5`

## 4. Generating Structured Responses with an LLM (RAG Retrieval)

In this final step, we use a Large Language Model (LLM) to analyze the clinical notes retrieved in the previous section. By injecting these relevant notes into a structured prompt, we enable the LLM to generate clinically useful, structured responses.

This completes the Retrieval-Augmented Generation (RAG) pipeline, connecting search results to intelligent language generation.

### Key Steps

1. **Create a Prompt Template (Step 4.1)**
   - Defines a reusable prompt structure that includes placeholders for both the query and the retrieved clinical context.
   - Specifies a structured output format, including metadata fields such as patient ID, encounter date, and a clinical summary.

2. **Invoke the LLM with Retrieved Context (Step 4.2)**
   - Fills the prompt with retrieved notes and the clinical query.
   - Sends the prompt to a local LLM (e.g., Qwen2 or LLaMA 3 via Ollama).
   - Returns a structured, context-aware response to the clinical question.

### Why It Matters

This generation step demonstrates the power of combining semantic search with generative AI:
- Produces context-rich answers grounded in patient data
- Supports clinical summarization and decision support
- Enables zero-shot reasoning over real-world clinical notes

<img src="./images/rag_generation.png" alt="RAG Generation" width="1250">


In [7]:
# -----------------------------------------------------------
# 4.1. Prompt Template: Dementia Phenotype from Retrieved Notes
# -----------------------------------------------------------
# This chat prompt guides the LLM to generate structured clinical summaries
# from notes retrieved via similarity search.

# Placeholders:
# - {retrieved_docs}: Injects top-matching clinical notes
# - {query}: A user-defined clinical question

# Output Expectations:
# - One structured summary per patient
# - Metadata included for traceability
# - Response based only on the most recent note per patient
# -----------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", f"""
You are a clinical phenotyping assistant.
Use ONLY the provided notes. Do NOT infer or invent.
Do NOT mention privacy/redaction/date shifting/removed identifiers.
"""),

    ("human", f"""
Retrieved notes:

{{retrieved_docs}}

Clinical question: {{query}}

For each relevant patient, provide:

Return a structured answer using bullet points:
1) Dementia Phenotype (Yes/No)
2) Evidence: up to 3 short direct quotes/phrases from the notes (or 'None')
3) Rationale: 1–2 sentences grounded in the evidence
4) Confidence: low/medium/high

Rules for Dementia Phenotype:
- Answer 'Yes' ONLY if dementia is explicitly documented (e.g., 'dementia', 'Alzheimer',
  'vascular dementia', 'Lewy body dementia') or clearly stated as a prior diagnosis.
- If dementia is not explicitly mentioned, answer 'No' (do not infer from memory complaints alone).
""")
])

print("✅ Prompt created and ready to use.")


✅ Prompt created and ready to use.


In [8]:
# -----------------------------------------------------------
# 4.2. Invoke LLM Using ALL Threshold-Retrieved Notes (with Traceability)
# -----------------------------------------------------------

from langchain_ollama import ChatOllama
from IPython.display import display, Markdown

docs_for_llm = results  # use retrieved results from 3.4

if len(docs_for_llm) == 0:
    display(Markdown("### No notes passed the threshold — nothing to send to the LLM."))
else:
    retrieved_blocks = []
    for rank, doc in enumerate(docs_for_llm, start=1):
        meta = doc.metadata or {}
        text = (doc.page_content or "")

        retrieved_blocks.append(
            f"---\n"
            f"[Retrieved Note #{rank}]\n"
            f"patient_num={meta.get('patient_num','N/A')} | visit_id={meta.get('visit_id','N/A')} | note_csn_id={meta.get('note_csn_id','N/A')}\n"
            f"note_type={meta.get('inpatient_note_type_dsc','N/A')} | create_dts_shifted={meta.get('create_dts_shifted','N/A')}\n\n"
            f"{text}\n"
        )

    retrieved_context = "\n".join(retrieved_blocks)

    display(Markdown(f"### Retrieved notes sent to LLM: {len(docs_for_llm)}"))
    display(Markdown(retrieved_context))

    final_prompt = prompt_template.format(
        retrieved_docs=retrieved_context,
        query=query
    )

    model = ChatOllama(model="qwen2", temperature=0.0)
    response = model.invoke(final_prompt)

    display(Markdown("### 🧠 LLM Output (Dementia Phenotype)"))
    display(Markdown(response.content))

### Retrieved notes sent to LLM: 5

---
[Retrieved Note #1]
patient_num=H112686715 | visit_id=6447865553 | note_csn_id=3925256583
note_type=Progress Notes | create_dts_shifted=N/A

A physician summary form including the medication list and diagnosis was sent on 8/24/2017. The documentation provides confirmation of a prior diagnosis, which is consistent with a working diagnosis of dementia.

---
[Retrieved Note #2]
patient_num=H115124574 | visit_id=6368409967 | note_csn_id=3075125799
note_type=Progress Notes | create_dts_shifted=N/A

The patient was admitted for observation with the initial assessment taking place on 8/18/2017. There was successful communication with the inpatient care manager regarding the admission, and congestive heart failure was documented as a relevant concern. Notably, there is a history suggesting possible cognitive impairment consistent with a prior diagnosis of dementia. No additional interval events or updates were provided.

---
[Retrieved Note #3]
patient_num=H113694308 | visit_id=6412697495 | note_csn_id=3578176631
note_type=Progress Notes | create_dts_shifted=N/A

The patient was observed sitting upright in bed and appeared comfortable. Vital signs were stable while breathing room air. No complaints of pain were reported at this time, but a cough was noted. A chest x-ray was performed at the bedside. The patient was provided with an update about the plan of care. Additional information can be found in the chart and recent orders.

A prior diagnosis of dementia continues to be considered in the ongoing assessment.

---
[Retrieved Note #4]
patient_num=H117604331 | visit_id=6403897423 | note_csn_id=3470839149
note_type=Progress Notes | create_dts_shifted=N/A

Today, the patient was seen in person for a follow-up visit and to receive a flu vaccination. Reports from her daughter indicate that there have been no new concerns and her condition remains stable. The patient continues to receive assistance at home with her activities of daily living. The care team will remain available for ongoing support and plans to check in by phone in approximately 3 months. There is a history of cognitive impairment consistent with a highly confident diagnosis of dementia.

---
[Retrieved Note #5]
patient_num=H114980041 | visit_id=6341343803 | note_csn_id=2916158553
note_type=Assessment & Plan Note | create_dts_shifted=N/A

There have been no recent diagnostic tests. The plan is to obtain Humphrey Visual Field testing and optical coherence tomography of the optic nerves to further evaluate the patient's condition, which includes a prior diagnosis of dementia.


### 🧠 LLM Output (Dementia Phenotype)

1) Dementia Phenotype (Yes/No)
2) Evidence: up to 3 short direct quotes/phrases from the notes (or 'None')
3) Rationale: 1–2 sentences grounded in the evidence
4) Confidence: low/medium/high

Patient H112686715:
- Dementia Phenotype: Yes
- Evidence: "The documentation provides confirmation of a prior diagnosis, which is consistent with a working diagnosis of dementia."
- Rationale: The note explicitly mentions that there is confirmation of a prior diagnosis consistent with dementia.
- Confidence: High

Patient H115124574:
- Dementia Phenotype: Yes
- Evidence: "Notably, there is a history suggesting possible cognitive impairment consistent with a prior diagnosis of dementia."
- Rationale: The note mentions that the patient has a history suggesting cognitive impairment consistent with a prior diagnosis of dementia.
- Confidence: Medium

Patient H113694308:
- Dementia Phenotype: Yes
- Evidence: "A prior diagnosis of dementia continues to be considered in the ongoing assessment."
- Rationale: The note states that dementia is being considered as part of the ongoing assessment, indicating a previous diagnosis.
- Confidence: Medium

Patient H117604331:
- Dementia Phenotype: Yes
- Evidence: "There is a history of cognitive impairment consistent with a highly confident diagnosis of dementia."
- Rationale: The note mentions that there is a history of cognitive impairment consistent with a highly confident diagnosis of dementia.
- Confidence: High

Patient H114980041:
- Dementia Phenotype: Yes
- Evidence: "There have been no recent diagnostic tests. The plan is to obtain Humphrey Visual Field testing and optical coherence tomography of the optic nerves to further evaluate the patient's condition, which includes a prior diagnosis of dementia."
- Rationale: The note indicates that there was a prior diagnosis of dementia as part of the ongoing assessment.
- Confidence: High

In [9]:
# -----------------------------------------------------------
# 4.3 (Bonus). Patient-Level Dementia Phenotyping from Notes
# -----------------------------------------------------------
# Features:
# - Processes patients sequentially
# - Early stop when dementia detected
# - Progress logging
# - Optional patient limit for faster runs
# -----------------------------------------------------------

import pandas as pd
from pathlib import Path
from langchain_ollama import ChatOllama

# Load evaluation labels (for later merge)
eval_df = pd.read_csv(Path("data_files") / "lab1_structured_dementia_eval.csv")

# LLM model
model = ChatOllama(model="qwen2", temperature=0.0)

# -----------------------------------------------------------
# OPTIONAL: limit number of patients (set to None to run all)
# -----------------------------------------------------------

PATIENT_LIMIT = 20   # change to None to process all patients
MAX_NOTES_PER_PATIENT = 5 # change to None to evaluate all notes per patient

unique_patients = notes_df["patient_num"].unique().tolist()

if PATIENT_LIMIT:
    unique_patients = unique_patients[:PATIENT_LIMIT]

total_patients = len(unique_patients)

print(f"Patients to process: {total_patients}\n")
print(f"Max notes per patient: {MAX_NOTES_PER_PATIENT}\n")

results = []

for idx, patient_num in enumerate(unique_patients, start=1):

    print(f"Processing patient {idx} of {total_patients} -> {patient_num}")

    patient_notes = notes_df[notes_df["patient_num"] == patient_num]

    # Keep only up to N notes per patient
    if MAX_NOTES_PER_PATIENT is not None:
        patient_notes = patient_notes.head(MAX_NOTES_PER_PATIENT)

    dementia_found = False
    notes_checked = 0
    first_yes_note_csn = None
    first_yes_visit_id = None
    evidence_preview = None

    for _, row in patient_notes.iterrows():

        notes_checked += 1

        note_text = (row["note_txt_deid"] or "").strip()
        if len(note_text) == 0:
            continue

        retrieved_docs = (
            f"patient_num={row.get('patient_num','N/A')} | visit_id={row.get('visit_id','N/A')} | "
            f"note_csn_id={row.get('note_csn_id','N/A')}\n"
            f"note_type={row.get('inpatient_note_type_dsc','N/A')} | "
            f"create_dts_shifted={row.get('create_dts_shifted','N/A')}\n\n"
            f"{note_text}"
        )

        query = "Does the patient have a documented diagnosis of dementia or Alzheimer disease?"

        final_prompt = prompt_template.format(
            retrieved_docs=retrieved_docs,
            query=query
        )

        response = model.invoke(final_prompt).content

        # simple detection
        if "Dementia Phenotype" in response and "Yes" in response.split("Dementia Phenotype", 1)[1][:60]:

            dementia_found = True
            first_yes_note_csn = row.get("note_csn_id")
            first_yes_visit_id = row.get("visit_id")
            evidence_preview = response[:400]

            break

    print(f"   dementia: {'YES' if dementia_found else 'NO'}")
    print(f"   notes evaluated: {notes_checked}\n")

    results.append({
        "patient_num": patient_num,
        "rag_dementia": dementia_found,
        "notes_checked": notes_checked,
        "first_yes_note_csn_id": first_yes_note_csn,
        "first_yes_visit_id": first_yes_visit_id,
        "rag_output_preview": evidence_preview
    })

rag_df = pd.DataFrame(results)

print("RAG results shape:", rag_df.shape)
display(rag_df.head(10))

Patients to process: 20

Max notes per patient: 5

Processing patient 1 of 20 -> H120513333
   dementia: NO
   notes evaluated: 5

Processing patient 2 of 20 -> H111336931
   dementia: NO
   notes evaluated: 5

Processing patient 3 of 20 -> H120897999
   dementia: NO
   notes evaluated: 5

Processing patient 4 of 20 -> H122074591
   dementia: YES
   notes evaluated: 1

Processing patient 5 of 20 -> H122355386
   dementia: NO
   notes evaluated: 5

Processing patient 6 of 20 -> H120189926
   dementia: NO
   notes evaluated: 5

Processing patient 7 of 20 -> H113566534
   dementia: NO
   notes evaluated: 5

Processing patient 8 of 20 -> H113383484
   dementia: NO
   notes evaluated: 5

Processing patient 9 of 20 -> H114032585
   dementia: YES
   notes evaluated: 2

Processing patient 10 of 20 -> H119829300
   dementia: NO
   notes evaluated: 5

Processing patient 11 of 20 -> H114744177
   dementia: YES
   notes evaluated: 2

Processing patient 12 of 20 -> H115124574
   dementia: YES
   no

,patient_num,rag_dementia,notes_checked,first_yes_note_csn_id,first_yes_visit_id,rag_output_preview
0,H120513333,False,5,NaN,NaN,NaN
1,H111336931,False,5,NaN,NaN,NaN
2,H120897999,False,5,NaN,NaN,NaN
3,H122074591,True,1,2.455733e+09,6.270645e+09,"1) Dementia Phenotype: Yes\n2) Evidence: ""prio..."
4,H122355386,False,5,NaN,NaN,NaN
5,H120189926,False,5,NaN,NaN,NaN
6,H113566534,False,5,NaN,NaN,NaN
7,H113383484,False,5,NaN,NaN,NaN
8,H114032585,True,2,1.896885e+09,6.272995e+09,"1) Dementia Phenotype: Yes\n2) Evidence: ""In l..."
9,H119829300,False,5,NaN,NaN,NaN


In [11]:
# -----------------------------------------------------------
# 4.4 (Bonus). Merge RAG phenotype with structured + gold labels
# -----------------------------------------------------------

merged = eval_df.merge(rag_df, on="patient_num", how="inner")

print("Merged shape:", merged.shape)
display(merged.head(20))

# Simple cross-tab vs gold
conf = pd.crosstab(
    merged["rag_dementia"].astype(bool),
    merged["gold_dementia"].astype(bool),
    rownames=["RAG Dementia"],
    colnames=["Gold Dementia"]
)
display(conf)

Merged shape: (20, 11)


,patient_num,dx_dementia_flag,med_dementia_flag,baseline_dementia,gold_diagnosis,gold_dementia,rag_dementia,notes_checked,first_yes_note_csn_id,first_yes_visit_id,rag_output_preview
0,H122355386,False,False,False,MCI vs. dementia,False,False,5,NaN,NaN,NaN
1,H119661286,False,False,False,No MCI or dementia diagnosis (normal),False,False,5,NaN,NaN,NaN
2,H113681764,False,False,False,No MCI or dementia diagnosis (normal),False,False,5,NaN,NaN,NaN
3,H122074591,False,False,False,Dementia,True,True,1,2.455733e+09,6.270645e+09,"1) Dementia Phenotype: Yes\n2) Evidence: ""prio..."
4,H114032585,False,False,False,Dementia,True,True,2,1.896885e+09,6.272995e+09,"1) Dementia Phenotype: Yes\n2) Evidence: ""In l..."
5,H113703318,False,False,False,Dementia,True,True,1,1.889879e+09,6.272766e+09,"1) Dementia Phenotype: Yes\n2) Evidence: ""Ther..."
6,H111336931,False,False,False,No MCI or dementia diagnosis (normal),False,False,5,NaN,NaN,NaN
7,H115648446,False,False,False,Dementia,True,True,1,2.998026e+09,6.283512e+09,"1) Dementia Phenotype: Yes\n2) Evidence: ""She ..."
8,H117670807,False,False,False,No MCI or dementia diagnosis (normal),False,False,5,NaN,NaN,NaN
9,H120513333,False,False,False,No MCI or dementia diagnosis (normal),False,False,5,NaN,NaN,NaN


Gold Dementia,False,True
RAG Dementia,,
False,11,1
True,1,7
